``CatBoost Algorithem``

Cat boost algorithm is a gradient boosting algorithm that uses categorical features without the need for extensive preprocessing. It is designed to handle categorical data efficiently and can be used for classification and regression tasks.

In [1]:
%pip install catboost -q

^C
Note: you may need to restart the kernel to use updated packages.


In [3]:
# library import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer

In [7]:
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [8]:
# pre processing 
# impute missing values using knn in age

imputer = KNNImputer(n_neighbors=5)
df['age'] = imputer.fit_transform(df[['age']])

# impute missing value using pandas in embarked and embark_town

df['embark_town']= df['embark_town'].fillna(df['embark_town'].mode()[0], inplace=True)
df['embarked']= df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)

df.drop('deck', axis=1, inplace=True)

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          891 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     891 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  embark_town  891 non-null    str     
 12  alive        891 non-null    str     
 13  alone        891 non-null    bool    
dtypes: bool(2), category(1), float64(2), int64(4), str(5)
memory usage: 79.4 KB


C:\Users\USER\AppData\Local\Temp\ipykernel_9136\1037817660.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['embark_town']= df['embark_town'].fillna(df['embark_town'].mode()[0], inplace=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_9136\1037817660.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame o

In [10]:
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_features

df[cat_features] = df[cat_features].astype('category')

C:\Users\USER\AppData\Local\Temp\ipykernel_9136\1171659959.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()


In [11]:
# split the dataset in x and y 

y = df['survived']
x = df.drop('survived', axis=1)

# train test split 

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)



In [ ]:
# run the catboost model
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=3,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=False,
)
model.fit(x_train, y_train)

# train the model 

model.fit(x_train, y_train, cat_features=cat_features)

# prediction 
y_pred = model.predict(x_test)

# evaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")